In [24]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import math
import torch

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

/kaggle/input/competitions/penyisihan-ifest-2026-dac/sample_submission.csv
/kaggle/input/competitions/penyisihan-ifest-2026-dac/train.csv
/kaggle/input/competitions/penyisihan-ifest-2026-dac/test.csv
Using: cpu


In [26]:
%pip install -q rapidfuzz==3.14.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.6 MB/s eta 0:00:00:00:01
Note: you may need to restart the kernel to use updated packages.


In [47]:
import os
import re
import json
import time
import hashlib
import platform
import collections
from copy import deepcopy

from pathlib import Path
from difflib import SequenceMatcher

os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"

import joblib
import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from rapidfuzz.distance import LCSseq
from rapidfuzz.fuzz import partial_ratio, ratio, token_sort_ratio

from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import normalize

from IPython.display import FileLink, display

fuzzy_ratio = ratio

WORK = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path.cwd()
)
os.chdir(WORK)

In [31]:
BASE_CONFIG = {
    "specs": {
        "lgb_cond7": {
            "kind": "lgb",
            "config": [7, 800, 0.03, 35, 5],
            "feature_mode": "custom",
            "conditional": True,
            "seeds": [42, 137],
        },
        "lgb_cond31": {
            "kind": "lgb",
            "config": [31, 800, 0.03, 35, 10],
            "feature_mode": "custom",
            "conditional": True,
            "seeds": [42, 137],
        },
        "lgb_all31": {
            "kind": "lgb",
            "config": [31, 800, 0.03, 35, 10],
            "feature_mode": "all",
            "conditional": True,
            "seeds": [42, 137],
        },
    },
    "names": [
        "lgb_cond7",
        "lgb_cond31",
        "lgb_all31",
    ],
    "weights": [0.25, 0.50, 0.25],
    "threshold": 0.7799999999999997,
}

V4_CONFIG = {
    "version": 4,
    "text_threshold": 0.70,
    "reference_threshold": 0.60,
    "reference_models": ["lgb15", "extra"],
    "reference_weights": [0.50, 0.50],
    "exact_pair_rule": True,
    "training_sha256": (
        "a4ba91606302455da8a9523906a25361d581ac15b108964cdb3ab6beb0adb5d8"
    ),
}

In [2]:
path = "/kaggle/input/competitions/penyisihan-ifest-2026-dac/"

train = pd.read_csv(path + "train.csv")
test = pd.read_csv(path + "test.csv")
sample_submission = pd.read_csv(path + "sample_submission.csv")


target = 'addicted_label'

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)
train.head()

Train Shape: (14397, 4)
Test Shape: (3603, 3)


,id,title,content,label
0,tr00000,Tak Lagi Jadi Jubir Yurianto Tak Umumkan Updat...,Ada perubahan besar besaran dalam pemaparan da...,0
1,tr00001,Aturan Swab PCR Bagi Penumpang Pesawat ke Kalb...,Aturan Swab PCR Bagi Penumpang Pesawat ke Kalb...,1
2,tr00002,Tim Riset Vaksin Sinovac Siapkan Laporan Perda...,Tim riset dari Fakultas Kedokteran Universitas...,1
3,tr00003,Kasus Corona Meksiko Kini Lebih dari 500 Ribu ...,Meksiko melaporkan kasus infeksi virus corona ...,1
4,tr00004,Menag Di Negara Pancasila Tak Ada Diktator Ma...,Menteri Agama Menag Yaqut Cholil Qoumas menj...,1


In [3]:
print("Shape train:", train.shape)
print("Kolom train:", train.columns.tolist())

print("\nTipe data:")
print(train.dtypes)

print(train["label"].value_counts())

print("Missing value:")
print(train.isna().sum())

Shape train: (14397, 4)
Kolom train: ['id', 'title', 'content', 'label']

Tipe data:
id         object
title      object
content    object
label       int64
dtype: object
label
1    12960
0     1437
Name: count, dtype: int64
Missing value:
id         0
title      0
content    0
label      0
dtype: int64


In [4]:
exact_duplicates = train[
    train.duplicated(
        subset=["title", "content", "label"],
        keep=False
    )
].sort_values(["title", "content"])

print("Jumlah baris duplikat identik:", len(exact_duplicates))

Jumlah baris duplikat identik: 6


In [5]:
# Jumlah kata
train["title_word_count"] = (
    train["title"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

train["content_word_count"] = (
    train["content"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

display(
    train[
        ["title_word_count", "content_word_count"]
    ].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
)

display(
    train.groupby("label")[
        ["title_word_count", "content_word_count"]
    ].median()
)

,title_word_count,content_word_count
count,14397.000000,14397.000000
mean,10.071821,309.949781
std,1.966900,169.567216
min,3.000000,3.000000
50%,10.000000,276.000000
75%,11.000000,356.000000
90%,13.000000,466.000000
95%,13.000000,581.200000
99%,15.000000,930.080000
max,23.000000,4238.000000


,title_word_count,content_word_count
label,,
0,10.0,297.0
1,10.0,273.0


In [6]:
pair_summary = (
    train.groupby(["title", "content"])
    .agg(
        jumlah_baris=("label", "size"),
        jumlah_label=("label", "nunique"),
        label_list=("label", lambda x: sorted(x.unique().tolist()))
    )
    .reset_index()
)

pair_conflicts = pair_summary[
    pair_summary["jumlah_label"] > 1
]

print(
    "Pasangan title-content dengan label bertentangan:",
    len(pair_conflicts)
)

display(pair_conflicts.head(10))

Pasangan title-content dengan label bertentangan: 0


,title,content,jumlah_baris,jumlah_label,label_list


In [7]:
train_clean = (
    train.drop_duplicates(
        subset=["title", "content", "label"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Sebelum:", train.shape)
print("Sesudah:", train_clean.shape)
print("\nDistribusi label:")
print(train_clean["label"].value_counts())

Sebelum: (14397, 6)
Sesudah: (14394, 6)

Distribusi label:
label
1    12960
0     1434
Name: count, dtype: int64


In [8]:
#Tokenisasi, stopword, stemming sederhana, dan hash artikel tetap mengikuti kode asli.
STOP = set(
    (
        "yang dan di ke dari untuk pada dengan dalam ini itu adalah "
        "akan oleh karena juga atau sebagai agar tidak tak bukan belum "
        "sudah telah saat bisa dapat masih lebih hanya hingga sampai "
        "jadi menjadi tentang ada setelah sebelum antara para sebuah "
        "semua saya kita mereka tersebut seperti serta atas lalu hari "
        "orang tahun covid corona 19 com kumparan"
    ).split()
)

NEG = set(
    (
        "tak tidak bukan belum tanpa jangan enggan batal gagal "
        "larang dilarang menolak"
    ).split()
)

TOK = re.compile(r"\b\w+\b", re.UNICODE)


def tokens(s):
    return TOK.findall(str(s).lower())


def norm(s):
    return " ".join(tokens(s))


def grams(ts, n):
    return set(zip(*(ts[k:] for k in range(n))))


def div(a, b):
    return a / (b + 1e-9)


def stem(w):
    if len(w) <= 5:
        return w

    for suff in ["nya", "kan", "lah", "an"]:
        if w.endswith(suff) and len(w) - len(suff) >= 4:
            w = w[:-len(suff)]
            break

    for pref in [
        "meng", "meny", "peng", "peny",
        "mem", "men", "pem", "pen",
        "ber", "ter", "per", "di", "ke",
    ]:
        if w.startswith(pref) and len(w) - len(pref) >= 4:
            w = w[len(pref):]
            break

    return w


def stable_hash(s):
    return int.from_bytes(
        hashlib.md5(s.encode()).digest()[:8],
        "little",
    )


def document(r):
    tt = tokens(r.title)
    cc = tokens(r.content)
    cs = set(cc)

    return tt, cc, cs


def body_key(s):
    return hashlib.sha256(norm(s).encode()).hexdigest()

In [9]:
# Mempertahankan fitur panjang, overlap, angka, kemiripan ejaan, dan TF-IDF.
class PairFeatures:
    def __init__(self):
        self.names = []

    def fit(self, d):
        self.word = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=160000,
            sublinear_tf=True,
            dtype=np.float32,
        )
        self.word.fit(
            pd.concat([d.title, d.content], ignore_index=True)
        )

        self.char = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=3,
            max_features=90000,
            sublinear_tf=True,
            dtype=np.float32,
        )
        self.char.fit(
            pd.concat([d.title, d.content], ignore_index=True)
        )

        self.idf = dict(
            zip(
                self.word.get_feature_names_out(),
                self.word.idf_,
            )
        )
        self.default = float(max(self.word.idf_))

        return self

    def transform(self, d):
        rows = []
        names = []

        for rr, (_, r) in enumerate(d.iterrows()):
            t = str(r.title)
            c = str(r.content)

            tt = tokens(t)
            cc = tokens(c)

            vals = []
            ns = []

            def add(n, v):
                ns.append(n)
                vals.append(v)

            # Karakteristik judul dan isi.
            for label, s, ts in [
                ("title", t, tt),
                ("body", c, cc),
            ]:
                add(label + "_chars", np.log1p(len(s)))
                add(label + "_words", np.log1p(len(ts)))
                add(label + "_uniq", div(len(set(ts)), len(ts)))

                for n, pat in [
                    ("double_space", r" {2,}"),
                    ("whitespace", r"\s"),
                    ("nbsp", r"\xa0"),
                    ("punct", r"[^\w\s]"),
                    ("upper", r"[A-Z]"),
                    ("digits", r"\d"),
                    ("newline", r"\n"),
                ]:
                    cnt = len(re.findall(pat, s))

                    add(label + "_" + n, cnt)
                    add(
                        label + "_" + n + "_rate",
                        div(cnt, len(s)),
                    )

                add(
                    label + "_kumparan",
                    float("kumparan" in s.lower()),
                )
                add(
                    label + "_detik",
                    float("detik" in s.lower()),
                )

            add("word_length_ratio", div(len(tt), len(cc)))

            weights = np.array(
                [self.idf.get(x, self.default) for x in tt]
            )

            ss = set(tt)
            significant = [x for x in tt if x not in STOP]
            numbers = [x for x in tt if x.isdigit()]

            capitals = [
                x.lower()
                for x in TOK.findall(t)
                if x[0].isupper() and x.lower() not in STOP
            ]

            # Kecocokan judul pada beberapa panjang awal artikel.
            for lim in [30, 60, 100, 200, 400, 0]:
                seg = cc[:lim] if lim else cc
                cs = set(seg)
                prefix = "seg" + str(lim)

                match = np.array([x in cs for x in tt])

                add(
                    prefix + "_recall",
                    div(match.sum(), len(tt)),
                )
                add(
                    prefix + "_idf_recall",
                    div(
                        (match * weights).sum(),
                        weights.sum(),
                    ),
                )

                for label, ts in [
                    ("significant", significant),
                    ("number", numbers),
                    ("capital", capitals),
                ]:
                    missing = [x for x in ts if x not in cs]

                    add(
                        prefix + "_" + label + "_missing",
                        len(missing),
                    )
                    add(
                        prefix + "_" + label + "_recall",
                        div(len(ts) - len(missing), len(ts)),
                    )

                add(
                    prefix + "_jaccard",
                    div(len(ss & cs), len(ss | cs)),
                )

                for n in [2, 3, 4]:
                    tg = grams(tt, n)
                    cg = grams(seg, n)

                    add(
                        prefix + "_gram" + str(n),
                        div(len(tg & cg), len(tg)),
                    )

            # Kata judul yang tidak ditemukan pada isi.
            cs = set(cc)
            missing = [x for x in tt if x not in cs]

            mw = sorted(
                [self.idf.get(x, self.default) for x in missing],
                reverse=True,
            )

            add("missing_max_idf", mw[0] if mw else 0)
            add("missing_mean_idf", np.mean(mw) if mw else 0)
            add("missing_idf_sum", sum(mw))
            add(
                "missing_len_max",
                max(map(len, missing), default=0),
            )

            for n in range(1, 5):
                add(
                    "miss_idf_top" + str(n),
                    mw[n - 1] if len(mw) >= n else 0,
                )

            positions = [
                cc.index(x) if x in cs else len(cc) + 1
                for x in tt
            ]

            add(
                "match_pos_mean",
                div(
                    np.mean(positions) if positions else 0,
                    len(cc),
                ),
            )
            add(
                "match_pos_max",
                div(max(positions, default=0), len(cc)),
            )

            add(
                "prefix_sequence",
                SequenceMatcher(
                    None,
                    tt,
                    cc[:max(2 * len(tt), 30)],
                    autojunk=False,
                ).ratio(),
            )

            add("title_in_body", float(norm(t) in norm(c)))
            add("starts_title", float(norm(c).startswith(norm(t))))

            # Kecocokan terbaik pada jendela artikel.
            for win in [20, 40, 80]:
                cover = []
                idfcover = []

                for st in range(0, len(cc), max(1, win // 2)):
                    ws = set(cc[st:st + win])
                    mt = np.array([x in ws for x in tt])

                    cover.append(div(mt.sum(), len(tt)))
                    idfcover.append(
                        div(
                            (mt * weights).sum(),
                            weights.sum(),
                        )
                    )

                add(
                    "max_window" + str(win),
                    max(cover, default=0),
                )
                add(
                    "max_idf_window" + str(win),
                    max(idfcover, default=0),
                )

            # Kemiripan ejaan kata yang tidak cocok.
            sims = []

            for x in set(missing):
                if len(x) < 3:
                    continue

                candidates = [
                    y for y in cs
                    if abs(len(x) - len(y)) <= 2 and len(y) >= 3
                ]

                sims.append(
                    max(
                        (fuzzy_ratio(x, y) / 100 for y in candidates),
                        default=0,
                    )
                )

            add("missing_fuzzy_mean", np.mean(sims) if sims else 1)
            add("missing_fuzzy_min", min(sims, default=1))
            add("missing_fuzzy_high", sum(s > 0.75 for s in sims))

            rows.append(vals)
            names = ns

            if rr and rr % 3000 == 0:
                print("rows transformed", rr, flush=True)

        dense = np.array(rows, dtype=np.float32)

        # Cosine similarity TF-IDF.
        for label, v in [
            ("word", self.word),
            ("char", self.char),
        ]:
            a = v.transform(d.title)
            b = v.transform(d.content)

            for lim in [200, 600, 1200]:
                z = v.transform(d.content.str[:lim])
                cos = np.asarray(
                    a.multiply(z).sum(axis=1)
                ).ravel()

                dense = np.column_stack([dense, cos])
                names.append(label + "_cos" + str(lim))

            cos = np.asarray(
                a.multiply(b).sum(axis=1)
            ).ravel()

            dense = np.column_stack([dense, cos])
            names.append(label + "_cos_full")

        self.names = names

        return dense

In [10]:
# Tambaha fitur, Fitur kapitalisasi, stemming, angka, negasi, dan LSA 64 dimensi tetap dipakai.
class Extras:
    def fit(self, d, vectorizer):
        self.word = vectorizer

        all_count = collections.Counter()
        caps = collections.Counter()

        for s in d.content:
            ts = TOK.findall(s)

            all_count.update(x.lower() for x in ts)
            caps.update(
                x.lower() for x in ts if x[0].isupper()
            )

        self.entity = {
            w: div(caps[w], n) if n >= 5 else 0.0
            for w, n in all_count.items()
        }

        self.lsa = TruncatedSVD(
            n_components=64,
            n_iter=5,
            random_state=42,
        )
        self.lsa.fit(self.word.transform(d.content))

        return self

    def transform(self, d):
        rows = []
        names = []

        for _, r in d.iterrows():
            tt = tokens(r.title)
            cc = tokens(r.content)
            cs = set(cc)

            ts = [stem(x) for x in tt]
            bs = [stem(x) for x in cc]

            vals = []
            ns = []

            def add(n, v):
                ns.append(n)
                vals.append(v)

            # Kecocokan kata yang sering ditulis dengan kapital.
            ew = np.array([self.entity.get(x, 0) for x in tt])
            present = np.array([x in cs for x in tt])

            add(
                "entity_weighted_recall",
                div((ew * present).sum(), ew.sum()),
            )
            add(
                "entity_missing_weight",
                (ew * (~present)).sum(),
            )

            for cut in [0.4, 0.65, 0.9]:
                ent = [
                    x for x in tt
                    if self.entity.get(x, 0) >= cut
                ]
                miss = [x for x in ent if x not in cs]

                add("entity_missing" + str(cut), len(miss))
                add(
                    "entity_recall" + str(cut),
                    div(len(ent) - len(miss), len(ent)),
                )

            # Kecocokan setelah stemming.
            for limit in [60, 150, 0]:
                ss = set(bs[:limit] if limit else bs)

                add(
                    "stem_recall" + str(limit),
                    div(sum(x in ss for x in ts), len(ts)),
                )

                for k in [2, 3]:
                    a = grams(
                        [stem(x) for x in tt if x not in STOP],
                        k,
                    )
                    b = grams(
                        [
                            stem(x)
                            for x in (cc[:limit] if limit else cc)
                            if x not in STOP
                        ],
                        k,
                    )

                    add(
                        "stem_significant_ngram"
                        + str(limit) + "_" + str(k),
                        div(len(a & b), len(a)),
                    )

            title_num = [x for x in tt if x.isdigit()]
            body_num = [x for x in cc if x.isdigit()]

            add("title_number_count", len(title_num))
            add(
                "title_number_missing",
                sum(x not in body_num for x in title_num),
            )

            for k in [2, 3]:
                a = grams(title_num, k)
                b = grams(body_num, k)

                add(
                    "number_sequence" + str(k),
                    div(len(a & b), len(a)),
                )

            # Negasi pada bagian artikel yang paling cocok.
            title_neg = sum(x in NEG for x in tt)
            add("title_negation_count", title_neg)

            for length in [40, 80, 150]:
                title_sig = set(
                    ts[i]
                    for i, x in enumerate(tt)
                    if x not in STOP
                )
                best = (-1, 0, 0, 0)

                for start in range(
                    0, len(cc), max(1, length // 2)
                ):
                    words = cc[start:start + length]
                    ss = set(bs[start:start + length])

                    coverage = div(
                        len(title_sig & ss),
                        len(title_sig),
                    )
                    negs = sum(x in NEG for x in words)

                    if coverage > best[0]:
                        best = (
                            coverage,
                            negs,
                            start,
                            len(words),
                        )

                add("local_negations" + str(length), best[1])
                add(
                    "local_negation_delta" + str(length),
                    title_neg - best[1],
                )
                add(
                    "local_negation_agree" + str(length),
                    float(bool(title_neg) == bool(best[1])),
                )

            candidates = set(
                ts[i]
                for i, x in enumerate(tt)
                if x not in STOP
            )

            negpred = {
                stem(cc[j])
                for i, x in enumerate(cc)
                if x in NEG
                for j in range(i + 1, min(len(cc), i + 5))
            }

            tpred = {
                stem(tt[j])
                for i, x in enumerate(tt)
                if x in NEG
                for j in range(i + 1, min(len(tt), i + 5))
            }

            add(
                "body_negated_title_stem",
                len(candidates & negpred),
            )
            add(
                "negated_stem_agreement",
                len(tpred & negpred),
            )
            add(
                "negated_title_unsupported",
                len(tpred - negpred),
            )

            rows.append(vals)
            names = ns

        extra = np.array(rows, dtype=np.float32)

        a = normalize(
            self.lsa.transform(self.word.transform(d.title))
        )
        b = normalize(
            self.lsa.transform(self.word.transform(d.content))
        )
        lead = normalize(
            self.lsa.transform(
                self.word.transform(d.content.str[:600])
            )
        )

        scalar = np.column_stack([
            (a * b).sum(axis=1),
            (a * lead).sum(axis=1),
        ])
        names += ["lsa_cos_body", "lsa_cos_lead"]

        latent = np.hstack([
            a,
            b,
            np.abs(a - b),
            a * b,
        ])
        names += [
            "lsa_" + str(i)
            for i in range(latent.shape[1])
        ]

        self.names = names

        return np.hstack([
            extra,
            scalar,
            latent,
        ]).astype(np.float32)


class ExtendedPairFeatures(PairFeatures):
    def fit(self, d):
        super().fit(d)
        self.extras = Extras().fit(d, self.word)

        return self

    def transform(self, d):
        a = super().transform(d)
        base_names = self.names.copy()

        e = self.extras.transform(d)
        self.names = base_names + self.extras.names

        return np.hstack([a, e])

In [11]:
class NewFeatures:
    def fit(self, d):
        self.counts = [
            collections.Counter(),
            collections.Counter(),
            collections.Counter(),
        ]

        self.groups = {}

        self.bucket_counts = [
            [collections.Counter() for _ in range(3)]
            for _ in range(5)
        ]

        self.bucket_totals = np.zeros((5, 2), dtype=int)
        self.totals = np.bincount(
            d.label,
            minlength=2,
        ).astype(int)

        df = collections.Counter()
        self.idf = {}

        for _, r in d.iterrows():
            tt, cc, cs = document(r)

            g = hashlib.sha256(
                norm(r.content).encode()
            ).hexdigest()
            bucket = stable_hash(g) % 5

            lab = int(r.label)
            ts = set(tt)

            df.update(cs)

            self.groups[g] = bucket
            gc = self.bucket_counts[bucket]
            self.bucket_totals[bucket, lab] += 1

            idx = 0 if lab == 1 else 1
            self.counts[idx].update(ts)
            gc[idx].update(ts)

            if lab == 1:
                self.counts[2].update(ts & cs)
                gc[2].update(ts & cs)

        self.idf = {
            w: np.log((len(d) + 1) / (n + 1)) + 1
            for w, n in df.items()
        }
        self.default = np.log(len(d) + 1) + 1

        return self

    def transform(self, d):
        rows = []
        names = []

        for ii, (_, r) in enumerate(d.iterrows()):
            tt, cc, cs = document(r)

            g = hashlib.sha256(
                norm(r.content).encode()
            ).hexdigest()

            bucket = self.groups.get(g)
            gc = (
                self.bucket_counts[bucket]
                if bucket is not None
                else None
            )
            tc = self.totals - (
                self.bucket_totals[bucket]
                if bucket is not None
                else 0
            )

            required = []
            logodds = []

            for w in tt:
                cnt = [
                    self.counts[k][w]
                    - (gc[k][w] if gc is not None else 0)
                    for k in range(3)
                ]

                required.append(
                    div(cnt[2] + 2, cnt[0] + 4)
                )
                logodds.append(
                    np.log(div(cnt[1] + 2, tc[0] + 4))
                    - np.log(div(cnt[0] + 2, tc[1] + 4))
                )

            required = np.array(required)
            logodds = np.array(logodds)

            iw = np.array([
                self.idf.get(w, self.default)
                for w in tt
            ])
            present = np.array([w in cs for w in tt])
            sig = np.array([w not in STOP for w in tt])

            vals = []
            ns = []

            def add(n, v):
                ns.append(n)
                vals.append(float(v))

            for label, mask in [
                ("all", np.ones(len(tt), dtype=bool)),
                ("missing", ~present),
                ("significant", sig),
                ("missing_sig", (~present) & sig),
            ]:
                for name, arr in [
                    ("req", required),
                    ("class_odds", logodds),
                ]:
                    aa = arr[mask]

                    add(
                        label + "_" + name + "_mean",
                        np.mean(aa) if len(aa) else 0,
                    )
                    add(
                        label + "_" + name + "_max",
                        np.max(aa) if len(aa) else 0,
                    )
                    add(
                        label + "_" + name + "_sum",
                        aa.sum(),
                    )
                    add(
                        label + "_" + name + "_min",
                        np.min(aa) if len(aa) else 0,
                    )

            for limit in [40, 80, 150, 0]:
                ss = set(cc[:limit] if limit else cc)
                m = np.array([w in ss for w in tt])
                weights = required * iw * sig

                add(
                    "required_recall" + str(limit),
                    div(
                        (m * weights).sum(),
                        weights.sum(),
                    ),
                )

            for cut in [0.5, 0.7, 0.85, 0.95]:
                add(
                    "required_missing" + str(cut),
                    np.sum(
                        (required >= cut)
                        & (~present)
                        & sig
                    ),
                )

            # Kecocokan urutan kata.
            ts = [stem(w) for w in tt if w not in STOP]
            bs = [stem(w) for w in cc if w not in STOP]

            add(
                "global_lcs",
                div(LCSseq.similarity(ts, bs), len(ts)),
            )
            add(
                "lead_lcs",
                div(LCSseq.similarity(ts, bs[:60]), len(ts)),
            )

            for length in [16, 32, 64]:
                scores = [
                    div(
                        LCSseq.similarity(
                            ts,
                            bs[s:s + length],
                        ),
                        len(ts),
                    )
                    for s in range(
                        0, len(bs), max(1, length // 2)
                    )
                ]

                add(
                    "max_ordered_lcs" + str(length),
                    max(scores, default=0),
                )

            nt = " ".join(tt)
            nb = " ".join(cc)
            st = " ".join(ts)
            sb = " ".join(bs)

            add("partial_raw_char", partial_ratio(nt, nb) / 100)
            add("partial_stem_char", partial_ratio(st, sb) / 100)

            # Kecocokan konteks di sekitar angka.
            num_context = []

            for i, w in enumerate(tt):
                if not w.isdigit():
                    continue

                a = set(
                    stem(z)
                    for z in tt[max(0, i - 3):i + 4]
                    if z not in STOP and not z.isdigit()
                )
                sc = []

                for j, z in enumerate(cc):
                    if z != w:
                        continue

                    b = set(
                        stem(q)
                        for q in cc[max(0, j - 4):j + 5]
                        if q not in STOP and not q.isdigit()
                    )

                    sc.append(div(len(a & b), len(a)))

                num_context.append(max(sc, default=0))

            add(
                "num_context_mean",
                np.mean(num_context) if num_context else 1,
            )
            add(
                "num_context_min",
                min(num_context, default=1),
            )
            add(
                "num_context_low",
                sum(z < 0.5 for z in num_context),
            )

            # Kesesuaian negasi pada konteks kata.
            body_polarity = collections.defaultdict(list)

            for j, w in enumerate(cc):
                if w not in STOP:
                    body_polarity[stem(w)].append(
                        float(
                            any(
                                z in NEG
                                for z in cc[max(0, j - 3):j]
                            )
                        )
                    )

            mismatches = []
            mismatch_weights = []
            consistent = []

            for i, w in enumerate(tt):
                if w in STOP:
                    continue

                bp = body_polarity.get(stem(w), [])

                if not bp:
                    continue

                pol = float(
                    any(
                        z in NEG
                        for z in tt[max(0, i - 3):i]
                    )
                )

                mismatch = abs(pol - np.mean(bp))

                mismatches.append(mismatch)
                mismatch_weights.append(iw[i] * required[i])
                consistent.append(
                    float(any(z == pol for z in bp))
                )

            add(
                "polarity_mismatch_mean",
                np.mean(mismatches) if mismatches else 0,
            )
            add(
                "polarity_mismatch_max",
                max(mismatches, default=0),
            )
            add(
                "polarity_weighted_mismatch",
                div(
                    np.dot(mismatches, mismatch_weights),
                    sum(mismatch_weights),
                ),
            )
            add(
                "polarity_no_consistent_count",
                len(consistent) - sum(consistent),
            )

            lexical_count = len(vals)

            rows.append(vals)
            names = ns

            if ii and ii % 3000 == 0:
                print("new features", ii, flush=True)

        self.names = names
        self.lexical_count = lexical_count

        return np.array(rows, dtype=np.float32)

In [13]:
# Mengukur hubungan label dengan kecocokan kat
class ConditionalFeatures:
    def fit(self, d):
        self.n = np.zeros(2, dtype=int)
        self.bn = np.zeros((5, 2), dtype=int)

        self.total = collections.Counter()
        self.bucket = [
            collections.Counter()
            for _ in range(5)
        ]
        self.groups = {}

        for _, r in d.iterrows():
            key = hashlib.sha256(
                norm(r.content).encode()
            ).hexdigest()
            b = stable_hash(key) % 5

            self.groups[key] = b

            lab = int(r.label)
            self.n[lab] += 1
            self.bn[b, lab] += 1

            ts = tokens(r.title)
            cs = set(tokens(r.content))

            events = [
                ("word", w, int(w in cs))
                for w in set(ts)
            ]
            events += [
                ("bigram", " ".join(z), 0)
                for z in set(zip(ts, ts[1:]))
            ]

            self.total.update(
                (lab, typ, w, m)
                for typ, w, m in events
            )
            self.bucket[b].update(
                (lab, typ, w, m)
                for typ, w, m in events
            )

        return self

    def transform(self, d):
        rows = []
        names = []

        for _, r in d.iterrows():
            ts = tokens(r.title)
            cs = set(tokens(r.content))

            key = hashlib.sha256(
                norm(r.content).encode()
            ).hexdigest()

            b = self.groups.get(key)
            bc = self.bucket[b] if b is not None else {}
            nc = self.n - (
                self.bn[b] if b is not None else 0
            )

            vals = []
            ns = []

            def add(n, v):
                ns.append(n)
                vals.append(float(v))

            def evidence(typ, w, m):
                counts = np.array(
                    [
                        self.total[lab, typ, w, m]
                        - bc.get((lab, typ, w, m), 0)
                        for lab in [0, 1]
                    ],
                    dtype=float,
                )

                weight = counts.sum() / (counts.sum() + 10)

                odds = (
                    np.log(div(counts[0] + 1, nc[0] + 2))
                    - np.log(div(counts[1] + 1, nc[1] + 2))
                ) * weight

                return odds, counts.sum()

            word_e = [
                evidence("word", w, int(w in cs))
                for w in ts
            ]

            ev = np.array([p[0] for p in word_e])
            freq = np.array([p[1] for p in word_e])

            mask_missing = np.array([w not in cs for w in ts])
            mask_sig = np.array([w not in STOP for w in ts])

            for label, mask in [
                ("all", np.ones(len(ts), dtype=bool)),
                ("missing", mask_missing),
                ("significant", mask_sig),
                (
                    "missing_significant",
                    mask_sig & mask_missing,
                ),
                (
                    "matched_significant",
                    mask_sig & (~mask_missing),
                ),
            ]:
                z = ev[mask]
                fr = freq[mask]

                add(
                    label + "_conditional_mean",
                    z.mean() if len(z) else 0,
                )
                add(label + "_conditional_sum", z.sum())
                add(
                    label + "_conditional_max",
                    max(z, default=0),
                )
                add(
                    label + "_conditional_min",
                    min(z, default=0),
                )
                add(
                    label + "_conditional_positive",
                    z[z > 0].sum(),
                )
                add(
                    label + "_conditional_negative",
                    z[z < 0].sum(),
                )
                add(
                    label + "_evidence_known",
                    np.sum(fr >= 5),
                )

            bg = [
                evidence("bigram", " ".join(z), 0)[0]
                for z in zip(ts, ts[1:])
            ]

            add(
                "bigram_evidence_mean",
                np.mean(bg) if bg else 0,
            )
            add(
                "bigram_evidence_max",
                max(bg, default=0),
            )
            add("bigram_evidence_sum", sum(bg))

            rows.append(vals)
            names = ns

        self.names = names

        return np.array(rows, dtype=np.float32)

In [15]:
# menggabungkan fitur
def transform_part(f, part):
    x = f.transform(part)
    return x, f.names


def parallel_transform(f, d, n_jobs=4):
    parts = [
        p for p in np.array_split(d, n_jobs)
        if len(p)
    ]

    results = joblib.Parallel(
        n_jobs=n_jobs,
        backend="loky",
    )(
        joblib.delayed(transform_part)(f, p)
        for p in parts
    )

    f.names = results[0][1]

    return np.vstack([x[0] for x in results])


def cols_for(spec, names):
    return [
        i
        for i, n in enumerate(names)
        if not (
            (
                spec["feature_mode"] == "custom"
                and n.startswith("lsa_")
                and not n.startswith("lsa_cos_")
            )
            or (
                not spec.get("conditional", True)
                and (
                    "_conditional_" in n
                    or "_evidence_known" in n
                    or n.startswith("bigram_evidence")
                )
            )
        )
    ]


def make_model(spec, seed):
    leaves, it, lr, ms, reg = spec["config"]

    return LGBMClassifier(
        n_estimators=it,
        num_leaves=leaves,
        learning_rate=lr,
        min_child_samples=ms,
        reg_lambda=reg,
        colsample_bytree=0.9,
        subsample=0.9,
        subsample_freq=1,
        n_jobs=4,
        verbosity=-1,
        random_state=seed,
    )


def fit_models(spec, A, y, names):
    cols = cols_for(spec, names)
    trained = []

    for seed in spec.get("seeds", [42]):
        m = make_model(spec, seed)
        m.fit(A[:, cols], y)

        trained.append(m)

    return trained, cols


def predict_models(trained, cols, B):
    return np.mean(
        [
            m.predict_proba(B[:, cols])[:, 1]
            for m in trained
        ],
        axis=0,
    )


class FeatureExtractorV2:
    def fit(self, d):
        self.base = ExtendedPairFeatures().fit(d)
        self.new = NewFeatures().fit(d)
        self.conditional = ConditionalFeatures().fit(d)

        return self

    def transform(self, d):
        base = parallel_transform(
            self.base,
            d,
            n_jobs=4,
        )
        a = self.new.transform(d)
        b = self.conditional.transform(d)

        self.names = (
            self.base.names
            + self.new.names
            + self.conditional.names
        )

        return np.hstack([
            base,
            a,
            b,
        ]).astype(np.float32)

In [17]:
#Membentuk 31 fitur referensi dari baris training dengan isi artikel yang sama, 
# termasuk kecocokan judul pada masing-masing label.
class ReferenceIndex:
    def fit(self, d):
        self.data = collections.defaultdict(list)

        for r in d.itertuples():
            self.data[body_key(r.content)].append(
                (norm(r.title), int(r.label))
            )

        return self

    def transform(self, d):
        rows = []

        for r in d.itertuples():
            refs = self.data.get(
                body_key(r.content),
                [],
            )

            t = norm(r.title)
            tt = t.split()

            ts = set(tt)
            st = set(stem(w) for w in tt)
            tn = set(w for w in tt if w.isdigit())
            tneg = set(tt) & NEG

            vals = [len(refs)]

            for lab in [0, 1]:
                rr = sorted(
                    set(s for s, y in refs if y == lab)
                )
                exact = sum(s == t for s in rr)

                vals.extend([len(rr), exact])

                best = []

                for s in rr:
                    ss = set(s.split())
                    sn = set(w for w in ss if w.isdigit())
                    sneg = ss & NEG

                    best.append([
                        ratio(t, s) / 100,
                        token_sort_ratio(t, s) / 100,
                        div(len(ts & ss), len(ts | ss)),
                        div(
                            len(st & set(stem(w) for w in ss)),
                            len(st | set(stem(w) for w in ss)),
                        ),
                        len(ts - ss),
                        len(ss - ts),
                        len((ts - ss) - STOP),
                        len((ss - ts) - STOP),
                        len(tn - sn),
                        len(sn - tn),
                        len(tneg - sneg),
                        len(sneg - tneg),
                        len(tt) - len(s.split()),
                    ])

                vals.extend(
                    max(best, key=lambda x: x[0])
                    if best
                    else [0] * 13
                )

            rows.append(vals)

        return np.asarray(rows, dtype=np.float32)

In [18]:
# Menambahkan dukungan isi artikel terhadap kata yang berubah
class ReferenceIndexPlus(ReferenceIndex):
    def transform(self, d):
        base = super().transform(d)
        rows = []

        for r in d.itertuples():
            refs = self.data.get(
                body_key(r.content),
                [],
            )

            tt = tokens(r.title)
            cc = tokens(r.content)

            ts = set(tt)
            cs = set(cc)
            lead = set(cc[:90])

            vals = []

            for label in [0, 1]:
                rr = sorted(
                    set(s for s, y in refs if y == label)
                )

                if rr:
                    ref = max(
                        rr,
                        key=lambda s: ratio(norm(r.title), s),
                    )

                    rs = set(ref.split())

                    left = ts - rs
                    right = rs - ts

                    sl = left - STOP
                    sr = right - STOP

                    vals += [
                        len(sl - cs),
                        len(sr - cs),
                        len(sl & cs),
                        len(sr & cs),
                        len(sl - lead),
                        len(sr - lead),
                        len(sl & lead),
                        len(sr & lead),
                        div(
                            LCSseq.similarity(tt, ref.split()),
                            max(len(tt), len(ref.split())),
                        ),
                        len(
                            set(stem(w) for w in sl)
                            - set(stem(w) for w in cc)
                        ),
                        len(
                            set(stem(w) for w in sr)
                            - set(stem(w) for w in cc)
                        ),
                    ]
                else:
                    vals += [0] * 11

            vals += [
                len((ts - STOP) - cs),
                len((ts - STOP) - lead),
                float("  " in str(r.title)),
                float("\xa0" in str(r.title)),
                float("  " in str(r.content)),
                float("\xa0" in str(r.content)),
            ]

            rows.append(vals)

        return np.hstack([
            base,
            np.asarray(rows, dtype=np.float32),
        ])


def cross_features(d, seed=731):
    X = np.zeros((len(d), 59), dtype=np.float32)

    for tr, va in StratifiedKFold(
        5,
        shuffle=True,
        random_state=seed,
    ).split(d, d.label):
        X[va] = (
            ReferenceIndexPlus()
            .fit(d.iloc[tr])
            .transform(d.iloc[va])
        )

    return X


def model_for(name):
    if name == "extra":
        return ExtraTreesClassifier(
            n_estimators=400,
            min_samples_leaf=3,
            max_features=0.8,
            n_jobs=2,
            random_state=42,
        )

    return LGBMClassifier(
        n_estimators=400,
        num_leaves=int(name[3:]),
        learning_rate=0.03,
        min_child_samples=20,
        reg_lambda=15,
        n_jobs=2,
        random_state=42,
        verbosity=-1,
    )

In [19]:
class EnhancedReferenceModel:
    def fit(self, d, names=("lgb15", "extra")):
        d = d.reset_index(drop=True)

        X = cross_features(d)
        active = X[:, 0] > 0

        self.models = []

        for n in names:
            m = model_for(n)
            m.fit(
                X[active],
                d.label.to_numpy()[active],
            )

            self.models.append(m)

        self.index = ReferenceIndexPlus().fit(d)

        return self

    def predict(self, d):
        X = self.index.transform(d)

        p = np.mean(
            [
                m.predict_proba(X)[:, 1]
                for m in self.models
            ],
            axis=0,
        )

        return (
            p,
            X[:, 0] > 0,
            (X[:, 2] > 0) & (X[:, 17] == 0),
            (X[:, 17] > 0) & (X[:, 2] == 0),
        )


def decisions(
    text,
    reference,
    active,
    exact0,
    exact1,
    config,
):
    lab = (
        text >= config["text_threshold"]
    ).astype(int)

    lab[active] = (
        reference[active] >= config["reference_threshold"]
    ).astype(int)

    if config["exact_pair_rule"]:
        lab[exact0] = 0
        lab[exact1] = 1

    return lab


class PipelineV4:
    def fit(
        self,
        d,
        config,
        base_config,
        base_pipeline=None,
    ):
        self.config = config

        if base_pipeline is None:
            f = FeatureExtractorV2().fit(d)
            A = f.transform(d)

            models = []

            for name, w in zip(
                base_config["names"],
                base_config["weights"],
            ):
                if w > 0:
                    ms, cols = fit_models(
                        base_config["specs"][name],
                        A,
                        d.label.to_numpy(),
                        f.names,
                    )
                    models.append((name, w, ms, cols))

            base_pipeline = {
                "feature_extractor": f,
                "models": models,
                "config": base_config,
            }

        self.base = base_pipeline
        self.references = EnhancedReferenceModel().fit(
            d,
            config["reference_models"],
        )

        return self

    def predict(self, d):
        X = self.base["feature_extractor"].transform(d)

        pt = sum(
            w * predict_models(ms, cols, X)
            for _, w, ms, cols in self.base["models"]
        )

        pr, active, e0, e1 = self.references.predict(d)

        return (
            decisions(
                pt,
                pr,
                active,
                e0,
                e1,
                self.config,
            ),
            pt,
            pr,
            active,
        )

In [20]:
def validate_v4(
    data,
    config,
    base_config,
    out_dir,
    seed=909,
):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    y = data.label.to_numpy()

    p_text = np.zeros(len(data))
    p_reference = np.zeros(len(data))
    p_reference_v3 = np.zeros(len(data))

    active = np.zeros(len(data), dtype=bool)
    exact0 = active.copy()
    exact1 = active.copy()

    fold_ids = np.zeros(len(data), dtype=int)

    folds = StratifiedKFold(
        3,
        shuffle=True,
        random_state=seed,
    )

    for fold, (tr, va) in enumerate(folds.split(data, y)):
        start = time.time()

        fold_train = data.iloc[tr].reset_index(drop=True)
        fold_valid = data.iloc[va].reset_index(drop=True)

        print(
            f"Validasi fold {fold + 1}/3",
            flush=True,
        )

        # Seluruh fitur dipelajari dari training fold.
        extractor = FeatureExtractorV2().fit(fold_train)

        A = extractor.transform(fold_train)
        B = extractor.transform(fold_valid)

        for name, weight in zip(
            base_config["names"],
            base_config["weights"],
        ):
            models, cols = fit_models(
                base_config["specs"][name],
                A,
                y[tr],
                extractor.names,
            )

            p_text[va] += weight * predict_models(
                models,
                cols,
                B,
            )

        # Referensi validasi hanya berasal dari training fold.
        R = cross_features(fold_train)
        S = (
            ReferenceIndexPlus()
            .fit(fold_train)
            .transform(fold_valid)
        )

        mask = R[:, 0] > 0

        active[va] = S[:, 0] > 0
        exact0[va] = (
            (S[:, 2] > 0) & (S[:, 17] == 0)
        )
        exact1[va] = (
            (S[:, 17] > 0) & (S[:, 2] == 0)
        )

        for name in config["reference_models"]:
            model = model_for(name)
            model.fit(R[mask], y[tr][mask])

            p_reference[va] += (
                model.predict_proba(S)[:, 1]
                / len(config["reference_models"])
            )

        # Pembanding V3 sesuai notebook asli.
        old_model = LGBMClassifier(
            n_estimators=250,
            num_leaves=7,
            learning_rate=0.03,
            min_child_samples=25,
            reg_lambda=20,
            n_jobs=2,
            random_state=42,
            verbosity=-1,
        )

        old_model.fit(
            R[mask, :31],
            y[tr][mask],
        )

        p_reference_v3[va] = old_model.predict_proba(
            S[:, :31]
        )[:, 1]

        fold_ids[va] = fold

        np.savez(
            out / f"fold{fold}_predictions.npz",
            indices=va,
            p_text=p_text[va],
            p_reference_v4=p_reference[va],
            p_reference_v3=p_reference_v3[va],
            reference_used=active[va],
            exact0=exact0[va],
            exact1=exact1[va],
        )

        print(
            "Selesai:",
            round(time.time() - start),
            "detik",
            flush=True,
        )

        del extractor, A, B, R, S

    pred_v4 = decisions(
        p_text,
        p_reference,
        active,
        exact0,
        exact1,
        config,
    )

    pred_v3 = (p_text >= 0.78).astype(int)
    pred_v3[active] = (
        p_reference_v3[active] >= 0.50
    ).astype(int)

    report = {
        "seed": seed,
        "folds": 3,
        "configuration_frozen_before_check": True,
        "scope": (
            "Repeated row CV; artikel dapat muncul di fold berbeda. "
            "Tidak ada tuning ulang."
        ),
        "v3_macro_f1": float(
            f1_score(y, pred_v3, average="macro")
        ),
        "v4_macro_f1": float(
            f1_score(y, pred_v4, average="macro")
        ),
        "fold_scores": [
            {
                "fold": k,
                "v3": float(
                    f1_score(
                        y[fold_ids == k],
                        pred_v3[fold_ids == k],
                        average="macro",
                    )
                ),
                "v4": float(
                    f1_score(
                        y[fold_ids == k],
                        pred_v4[fold_ids == k],
                        average="macro",
                    )
                ),
            }
            for k in range(3)
        ],
        "v4_classification_report": classification_report(
            y,
            pred_v4,
            output_dict=True,
        ),
        "v4_confusion_matrix": confusion_matrix(
            y,
            pred_v4,
        ).tolist(),
        "config": config,
    }

    (out / "check_report.json").write_text(
        json.dumps(report, indent=2),
        encoding="utf-8",
    )

    pd.DataFrame({
        "id": data.id,
        "label": y,
        "fold": fold_ids,
        "p_text": p_text,
        "p_reference_v3": p_reference_v3,
        "p_reference_v4": p_reference,
        "reference_used": active,
        "prediction_v3": pred_v3,
        "prediction_v4": pred_v4,
    }).to_csv(
        out / "check_oof.csv",
        index=False,
    )

    return report

In [21]:
RUN_VALIDATION = False

if RUN_VALIDATION:
    validation_report = validate_v4(
        train_clean,
        V4_CONFIG,
        BASE_CONFIG,
        WORK / "check_seed909",
    )

    print(
        "V3 Macro F1:",
        validation_report["v3_macro_f1"],
    )
    print(
        "V4 Macro F1:",
        validation_report["v4_macro_f1"],
    )

    display(
        pd.DataFrame(validation_report["fold_scores"])
    )

In [33]:
# Training, Menghapus metadata hash dari konfigurasi lama.
V4_CONFIG.pop("training_sha256", None)

OUT_DIR = WORK / "results_v4"
OUT_DIR.mkdir(parents=True, exist_ok=True)

model = PipelineV4().fit(
    train_clean,
    V4_CONFIG,
    BASE_CONFIG,
)

joblib.dump(
    model,
    OUT_DIR / "trained_pipeline_v4.joblib",
    compress=3,
)

manifest = {
    "training_rows": len(train_clean),
    "config": V4_CONFIG,
    "base_config": BASE_CONFIG,
    "python": platform.python_version(),
}

(OUT_DIR / "training_manifest_v4.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("Training selesai:", len(train_clean), "baris")

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


rows transformed 3000
rows transformed 3000
rows transformed 3000
rows transformed 3000
new features 3000
new features 6000
new features 9000
new features 12000
Training selesai: 14394 baris


In [39]:
# cari threshold terbaik
from copy import deepcopy
from sklearn.model_selection import train_test_split

baseline_config = deepcopy(model.config)
baseline_base_config = deepcopy(model.base["config"])

# Pembagian seluruhnya berasal dari train_clean:
# 70% training, 15% kalibrasi, 15% pemeriksaan.
fit_data, validation_data = train_test_split(
    train_clean,
    test_size=0.30,
    stratify=train_clean["label"],
    random_state=2027,
)

calibration_data, check_data = train_test_split(
    validation_data,
    test_size=0.50,
    stratify=validation_data["label"],
    random_state=909,
)

validation_model = PipelineV4().fit(
    fit_data.reset_index(drop=True),
    deepcopy(baseline_config),
    baseline_base_config,
)


def evaluate_reference_thresholds(data, thresholds):
    _, pt, pr, active = validation_model.predict(data)

    R = validation_model.references.index.transform(data)
    exact0 = (R[:, 2] > 0) & (R[:, 17] == 0)
    exact1 = (R[:, 17] > 0) & (R[:, 2] == 0)

    rows = []

    for threshold in thresholds:
        config = deepcopy(baseline_config)
        config["reference_threshold"] = float(threshold)

        prediction = decisions(
            pt, pr, active, exact0, exact1, config
        )

        rows.append({
            "reference_threshold": float(threshold),
            "macro_f1": f1_score(
                data["label"],
                prediction,
                average="macro",
            ),
        })

    return pd.DataFrame(rows)


baseline_reference = float(
    baseline_config["reference_threshold"]
)

threshold_grid = np.unique(
    np.append(
        np.round(np.arange(0.50, 0.701, 0.01), 2),
        baseline_reference,
    )
)

# Pemilihan kandidat hanya menggunakan bagian kalibrasi.
calibration_scores = evaluate_reference_thresholds(
    calibration_data,
    threshold_grid,
)

calibration_scores["distance"] = (
    calibration_scores["reference_threshold"]
    - baseline_reference
).abs()

ranked_scores = calibration_scores.sort_values(
    ["macro_f1", "distance", "reference_threshold"],
    ascending=[False, True, True],
)

best_reference = float(
    ranked_scores.iloc[0]["reference_threshold"]
)

# Bagian pemeriksaan hanya membandingkan baseline dan kandidat terpilih.
check_scores = evaluate_reference_thresholds(
    check_data,
    np.unique([baseline_reference, best_reference]),
).set_index("reference_threshold")["macro_f1"]

calibration_lookup = calibration_scores.set_index(
    "reference_threshold"
)["macro_f1"]

improved = (
    calibration_lookup.loc[best_reference]
    > calibration_lookup.loc[baseline_reference]
) and (
    check_scores.loc[best_reference]
    > check_scores.loc[baseline_reference]
)

selected_config = deepcopy(baseline_config)

if improved:
    selected_config["reference_threshold"] = best_reference
    print("Kandidat diterima:", best_reference)
else:
    print("Tetap baseline:", baseline_reference)

display(pd.DataFrame({
    "Konfigurasi": ["Baseline", "Kandidat"],
    "Threshold": [baseline_reference, best_reference],
    "Macro F1 kalibrasi": [
        calibration_lookup.loc[baseline_reference],
        calibration_lookup.loc[best_reference],
    ],
    "Macro F1 pemeriksaan": [
        check_scores.loc[baseline_reference],
        check_scores.loc[best_reference],
    ],
}))

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


new features 3000
new features 6000
new features 9000


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/l

Kandidat diterima: 0.68


,Konfigurasi,Threshold,Macro F1 kalibrasi,Macro F1 pemeriksaan
0,Baseline,0.60,0.873915,0.850306
1,Kandidat,0.68,0.876038,0.850936


In [45]:
assert test.id.is_unique
assert sample_submission.id.is_unique
assert set(test.id) == set(sample_submission.id)

V4_CONFIG["reference_threshold"] = 0.68
V4_CONFIG["reference_weights"] = [0.50, 0.50]
model.config = deepcopy(V4_CONFIG)

label, p_text, p_reference, reference_used = model.predict(test)

print("Prediksi selesai:", len(label), "baris")

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


new features 3000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

Prediksi selesai: 3603 baris


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [46]:
submission = pd.DataFrame({
    "id": test["id"].to_numpy(),
    "label": label
})

assert submission["id"].is_unique
assert not submission.isna().any().any()
assert submission["label"].isin([0, 1]).all()


submission.to_csv(
    "/kaggle/working/submissionterus.csv",
    index=False
)

print("Submission siap!")

Submission siap!
